# PyA - AGen
agen is a submodule of pya that provides a simple and flexible way of generate audio signals using audio generators which lazily generate audio samples on demand.

The generators can be further combined with other generators (forming a tree of generator nodes) to create more complex
audio signals.

## Builtin AGens

### Control signal AGens

In [ ]:
from pya.agen.lib import Line, ADSR, XLine, Env
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 5))

Line(0, 1, 1).gen_asig().plot(label="Line")
XLine(10, 1, 1).gen_asig().plot(label="XLine")
Env(values=[2, 1.2, 3], dtimes=[0.5, 0.5]).gen_asig().plot(label="Env")
ADSR(
    attack=0.1,
    decay=0.1,
    sustain=0.5,
    release=0.1,
    level=0.8, 
).gen_asig().plot(label="ADSR")
plt.legend();

### Oscillators

In [ ]:
from pya.agen.lib import LFPulse, LFSaw, SinOsc, WhiteNoise, BLIT, BLSaw, BLPulse, BrownNoise

gens = [
    SinOsc(freq=10),
    LFPulse(freq=10),
    LFSaw(freq=10),
    WhiteNoise(),
    BrownNoise(), 
    BLIT(freq=10),
    BLSaw(freq=10),
    BLPulse(freq=20),
]
plt.figure(figsize=(20, 7))
for i, gen in enumerate(gens):
    (gen - 3 * i).gen_asig(seconds=2).plot(label=gen.label)

plt.legend();

### Filters

In [ ]:
from pya.agen.lib import LFilter, OnePole, OneZero, BPF, HPF, LPF, BRF

filters = [
    LFilter.p(
        b=[1, 0, 0, 0, 0], 
        a=[1, -0.5, 0.25, -0.125, 0.0625]
    ),
    OnePole.p(coef=-0.5), 
    OneZero.p(coef=0.5),
    BPF.p(f_0=10_000, bw=0.1), 
    HPF.p(freq=10_000), 
    LPF.p(freq=10_000),
    BRF.p(f_0=10_000, bw=0.1),
]

plt.figure(figsize=(20, 7))
for i, gen in enumerate(filters):
    ((SinOsc(freq=Line(20, 22050, 2)) | gen) - 3 * i).gen_asig().plot(label=gen.gen_class.__name__)
plt.legend();

### Spatialization

In [ ]:
from pya.agen.lib import Pan2, PanAz

plt.figure(figsize=(20, 5))

plt.subplot(1, 2, 1)
Pan2(WhiteNoise(), Line(-1, 1, 1)).gen_asig().plot(label="Pan2", offset=2)
plt.title("Pan2")

plt.subplot(1, 2, 2)
PanAz(4, WhiteNoise(), Line(-1, 1, 1)).gen_asig().plot(label="PanAz", offset=2)
plt.title("PanAz");

## Combining AGens

As you might have already seen in the examples, it is possible to combine multiple AGens to create more complex signals by supplying AGens as arguments to other AGens.

In [ ]:
SinOsc(freq=Line(0, 100, 1)).gen_asig().plot()

## Supported Operators
Various built-in Python operators are supported to combine AGens in different ways.

In [ ]:
(SinOsc(freq=10) + Line(0, 7, 1)).gen_asig().plot(label="Sum")
(SinOsc(freq=10) * Line(0, 10, 1)).gen_asig().plot(label="Mul")
(10 * SinOsc(freq=10)).gen_asig(seconds=1).plot(label="Mul with constant")
(SinOsc(freq=10) / Line(1, 7, 1)).gen_asig().plot(label="Div")
(Line(0, 10, 0.5) & Line(10, 0, 0.5)).gen_asig().plot(label="Concat")
plt.legend();

## Currying
It is possible to create partial AGens using the `AGen.p()` classmethod that can be completed by passing another AGen using the `|` operator for a possibly more human-readable way of creating complex AGens.

In [ ]:
gen = SinOsc(freq=Line(0, 100, 1), phase=0.5)
gen2 = Line(0, 100, 1) | SinOsc.p(phase=0.5) # Equivalent to the previous line
gen.gen_asig().plot()
gen2.gen_asig().plot();

## Visualizing AGens

To visualize the structure of an AGen, you can create a graphviz graph of the AGen by calling the `create_graph` method. This, however requires the (optional) `graphviz` package to be installed. 

In [ ]:
(SinOsc(freq=SinOsc(freq=Line(0, 100, 1))) * XLine(10, 1, 0.1).delay(seconds=0.9, padding="first")).create_graph()

## Duration of AGens
AGens can both have an infinite or limited duration. An AGen is considered as stopped when it does not
produce the required amount of samples in a block. 

When an AGen stops, it is possible to choose between three behaviors:
- `DoneAction.STOP` or "stop" (default) will simply terminate the generation of new samples.
- `DoneAction.LAST` or "last" will repeat the last sample indefinitely.
- `DoneAction.LOOP` or "loop" will restart the AGen from the beginning.

In [ ]:
Line(0, 1, 1, done="stop").gen_asig(seconds=3).plot(label="stop")
Line(1, 2, 1, done="loop").gen_asig(seconds=3).plot(label="loop")
Line(2, 3, 1, done="last").gen_asig(seconds=3).plot(label="last")
plt.legend();

## Multiple Channels
It is also possible to create AGens with multiple channels. For example to create a stereo signal
with two sine waves with different frequencies you can do the following:

In [ ]:
from pya.agen.core import stereo, multi_channel

SinOsc(freq=stereo(5, 20)).gen_asig(seconds=1).plot(offset=2.5);

If one node of an AGen has multiple channels this propagates up to the root node. However, 
it is not possible to mix AGens with different number of channels as it is not clear which
channel should be mapped to which.

If you would like to mix AGens with a different number of channels, you can increase the number of channels
with the `expand_channels()` function.
If you want to select a subset of the channels you can use the built-in indexing operator `[]`. 

In [ ]:
from pya.agen.lib import expand_channels

plt.figure(figsize=(15, 7))


foo = SinOsc(freq=stereo(5, 20)) # 2 channels
bar = SinOsc(freq=multi_channel(1, 2, 3)) # 3 channels

try:
    # This will raise an error because foo and bar have a different number of channels
    (foo + bar).gen_asig(seconds=1)
    raise Exception("This should not happen")
except ValueError as e:
    print(e)

plt.subplot(2, 2, 1)
# This works because we expand the number of channels of foo to match bar
(expand_channels(foo, target_count=3, strategy="cycle") + bar).gen_asig(seconds=1).plot(offset=2)
plt.title("expand_channels()")

plt.subplot(2, 2, 2)
# This also works because we only select the first two channels of bar
(foo + bar[:2]).gen_asig(seconds=1).plot(offset=2)
plt.title("Slicing")

plt.subplot(2, 2, 3)
# Or select the channels with a boolean mask
(foo + bar[[True, False, True]]).gen_asig(seconds=1).plot(offset=2)
plt.title("Boolean mask")

plt.subplot(2, 2, 4)
# Keep the first channel of bar and mix the rest
(foo + multi_channel(bar[0], bar[1:].mix())).gen_asig(seconds=1).plot(offset=2)
plt.title("Mixing channels");

# Loading Audio Files

With the `PlayAsig` AGen it is possible to play `Asig`s. There is also a shortcut for directly loading audio files with the `load_file` function.

In [ ]:
from pya.agen.lib import PlayAsig

PlayAsig.load_file("samples/ping.mp3").gen_asig().plot();

# Composing Sequences of AGens

To create a sequence of AGens, you can use the `SeqAGen` AGen. This AGen takes a list of AGens and onset times as arguments. For sequences with many AGens, it is here not recommend to use the `create_graph` method to visualize the structure of the AGen as this graph can become very large in this case. 
Instead, for this AGen we provide a `plot_sequence` function that creates timeline of the sequence.

In [ ]:
import matplotlib.pyplot as plt

from pya.agen.lib import SeqAGen

seq = SeqAGen([
    (0, SinOsc(freq=50).limit(seconds=1).with_label("Sine")), 
    (0.5, Line(0, 1, 1)),
    (1, WhiteNoise().limit(seconds=1).with_label("Noise")),
])

plt.figure(figsize=(20, 5))
plt.subplot(1, 2, 1)

seq.gen_asig().plot()

plt.subplot(1, 2, 2)
seq.plot_sequence()

# Creating Custom AGens

To create a custom AGens, you can subclass the `AGen` class and implement the `_generate_new` method.
In this method new samples are generated in blocks of size `sample_count`. 
The `generate_new` method will always be called for consecutive blocks of samples without any overlap or skipping samples.
If an AGen is reused multiple times in the same graph, the `generate_new` method will not be called multiple times for the same block of samples.
Instead, the samples are cached and reused.

To carry information across blocks and to access previous samples, you can use the `state` attribute of the AGen. 
An AGen has a separate state for each channel. 

It is possible to clear the state of an AGen by calling the `reset()` method. 

In [ ]:
from pya.agen.core import AGen, SingleChannelGen
import numpy as np


class CustomMultiChannel(AGen):
    def __init__(self, *args, **kwargs):
        super().__init__(
            *args, 
            channels=3, 
            **kwargs,
        )

    def _generate_new(
        self, 
        sample_count: int, # The amount of samples that should be generated
        start: int, # The index of the first sample
        channel: int, # The index of the channel for which the samples should be generated
    ) -> np.ndarray:
        # The AGen has a separate state for each channel
        block_num = self.state.data.get("block_num", 0)
        self.state.data["block_num"] = block_num + channel
        return np.full(sample_count, block_num)
    

class CustomSingleChannel(SingleChannelGen):
    def _generate_single(
        self, 
        sample_count: int, # The amount of samples that should be generated
        start: int, # The index of the first sample
    ) -> np.ndarray:
        block_num = self.state.data.get("block_num", 0)
        self.state.data["block_num"] = block_num + 1
        return np.full(sample_count, block_num)
    

plt.figure(figsize=(20, 5))
plt.subplot(1, 2, 1)
CustomMultiChannel().limit(seconds=1).gen_asig().plot(offset=3)

plt.subplot(1, 2, 2)
CustomSingleChannel().limit(seconds=1).gen_asig().plot();

`SingleChannelGen` is a subclass of `AGen` which simplifies the creation of single channel AGens.

## Custom AGens with Nodes

In [ ]:
from pya.agen.types import GenOrNum


class TestGen(SingleChannelGen):
    def __init__(self, a: GenOrNum, b: GenOrNum, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        
        # You can register nodes with self._add_node(gen, name)
        # This must come after the super constructor call since the super class must
        # be initialized before the nodes can be registered
        self._add_node(a, "a")

        # self._add_node() also returns the node so you can store it in a variable
        # if you prefer this method
        self.child_2 = self._add_node(b, "b", convert_num_to_arr=True)
        
    # Before _generate_new is called all nodes are sampled.
    # You can access the generated samples either by using the nodes dictionary
    # or by accessing the value property of the node
    # After _generate_new is finished the samples stored in the nodes will
    # be cleared
    def _generate_single(self, **_) -> np.ndarray:
        return self.nodes["a"] + self.child_2.value
    
class TestGen(AGen):
    def __init__(self, a: GenOrNum, b: GenOrNum, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Add the nodes as regular attributes
        self.a = a
        self.b = b

    # If we don't use the Node API, we have to implement get_nodes
    def get_nodes(self):
        return {
            "a": self.a,
            "b": self.b,
        }
    
    def _generate_new(self, sample_count, start, channel):
        # This samples from both nodes and also cuts the samples such that they have the same length
        _, (a, b) = self._get_samples_from_multiple([self.a, self.b], sample_count, start, channel)
        return a + b
    
    
TestGen(SinOsc(freq=1), Line(0, 10, 1)).gen_asig().plot()

TestGen(1, 2).gen_asig(seconds=1).plot()

TestGen(SinOsc(freq=1), Line(0, 10, 1)).create_graph()

# Metatools

To quickly create an AGen from an existing function, you can use the `x_gen` function to convert a function into an agen.

In [ ]:
from pya.agen.core import xgen
import numpy as np

xgen(
    np.floor, 
    # All arguments here are passed to the constructor the AGen
    label="Test", 
)(
    # Here you can pass the values that should be passed to the function
    # These can be both AGen instances or constants
    Line(0, 10, 1)
).gen_asig().plot();

If you specifically want to map a function to the output of an AGen, you can use the `apply` method.
Some common functions are already implemented and can be accessed directly.

In [ ]:
Line(0, 10, 1).apply(np.floor).gen_asig().plot()

# This also allows for easy chaining of functions
Line(0, 10, 1).apply(np.floor).apply(np.sqrt).gen_asig().plot()

Line(0, 10, 1).sqrt().arctan().sign().gen_asig().plot();

## Other useful functions

In [ ]:
gen = SinOsc(freq=10) * Line(0, 1, 2)

gen.delay(
    seconds=1,
    padding="first",  # Can be "first" or "zero" 
).gen_asig().plot(label="Delay")
gen.skip(seconds=1).gen_asig().plot(label="Skip")
plt.legend()
plt.show()

# Create a stereo AGen from a mono channel AGen
stereo_gen = gen.stereo()
print(stereo_gen.channels)



### Fading in and out

In [ ]:
gen = SinOsc(freq=50).limit(seconds=2)
gen.fade_in(0.2, curve=1).gen_asig().plot(label="Linear fade in")
(gen.fade_out(0.2, curve=1) - 3).gen_asig().plot(label="Linear fade out")
(gen.fade_in(0.2, curve=2) - 6).gen_asig().plot(label="Squared fade in")
(gen.fade(0.2, curve=2) - 9).gen_asig().plot(label="Squared fade in + out")
plt.legend()
plt.show()

# Sound Examples

In [ ]:
from pya import *
from pya.agen.core import *
from pya.agen.lib import *
from pyamapping import midi_to_cps
import matplotlib.pyplot as plt

s = Aserver.startup_default_server()

In [ ]:
%matplotlib widget
def fake_bass(freq: float, h=0.5, imax=5) -> AGen:
    i = ADSR(0.001, 0.1, 0.0, 0.5, done='last') * imax
    env = ADSR(0.05, 0.01, 0.01, 0.5, 0.8)
    return (SinOsc(freq + i * SinOsc(h * freq)) * env).with_label(f"FakeBass")

octaves = [12, 7, 10, 12, 10, 7, 5, 7]

gen = SeqAGen([
    (i * 0.25, fake_bass(freq=midi_to_cps(36 + p), h=0.5, imax=100))
    for i, p in enumerate(octaves * 3)
])


plt.figure(figsize=(10, 5))
gen.gen_asig().plot().stereo().play(onset=1)

In [ ]:
from pya.agen.lib import LPF, BPF

def synth(note):
    freq = midi_to_cps(note)
    return (BLSaw(freq=freq) + BLSaw(freq=freq * 1.2) * 0.4) * ADSR(0.1, 0.02, 0.05, 0.3) | LPF.p(freq=2000)

synth(40).gen_asig().play().plot()

In [ ]:
notes = [
    (0, [72, 60]), 
    (2, [60]),
    (4, [84, 72]), 
    (6, [84, 72, 60]), 
    (8, [75]), 
    (10, [87, 72]),
    (12, [72, 60]), 
    (14, [72, 84]), 
    (16, [68]), 
    (18, [80]), 
    (20, [68, 80]), 
    (22, [84, 68]), 
    (24, [67]), 
    (26, [67]), 
    (28, [70, 79]), 
    (30, [70, 84]), 
]

repetitions = 2
notes_new = []
for n in range(repetitions):
    notes_new += [(time + 32 * n, note) for time, note in notes]

notes = notes_new

SeqAGen([
    (time / 8, synth(n - 24)) 
    for time, note in notes
    for n in note
]).gen_asig().play(onset=0.5).plot()

# Upsampling AGens (Experimental)
It is also possible to combine AGens with different sample rates. 
When combining AGens with different sample rates, the AGens with the lower sample rate will be upsampled to the highest sample rate.
However, this feature is currently experimental and may not yield expected results.

In [ ]:
SinOsc(freq=Line(0, 10, 1, sr=100), sr=44100).gen_asig().plot()

Some AGens have an adaptive sample rate which means that the sample rate is automatically set to the highest sample rate of the children nodes. This requires at least one child node to be an AGen (and not a number).

In [ ]:
gen = SinOsc(
    freq=Line(0, 10, 1, sr=100), 
    phase=Line(0, 2 * np.pi, 1, sr=1000),
    sr=None,  # Set the sample rate to None to use an adaptive sample rate
)

print(f"Sample rate: {gen.sr}")

try:
    # This will raise an error because no sample rate is set due to none of the nodes
    # being AGens
    SinOsc(freq=1, phase=2, sr=None).gen_asig(seconds=1)
    print("This should not happen")
except ValueError as e:
    print(f"Got expected error: {e}")

This is used by, e.g., the AGens modelling the mathematical operators such as `AddGen`, `MulGen` or `PowGen`.

## Current Limitations of the Upsampling
When sampling signals with a finite sample rate each sample represents a certain time interval. 
However, since each sample is a scalar value, we need to choose a specific point in this time interval to represent the sample.
Here, we choose the start of the time interval to represent the sample. 

This means that, e.g., if we create a Line that goes from $0$ to $1$ in $1$ second with a sample rate of 5 Hz, the first sample will be $0$ and the last sample will not be $1$ but instead $0.8$ as this is the initial value of the interval represented by the last sample.

In [ ]:
import numpy as np
plt.step(np.linspace(0, 1, 6), np.concatenate([np.linspace(0, 0.8, 5), [0.8]]), where="post", c="gray", label="Signal")
plt.plot(np.linspace(0, 1, 10000), np.linspace(0, 1, 10000))
plt.plot(np.linspace(0, 0.8, 5), np.linspace(0, 0.8, 5), "o")

plt.ylim(bottom=-.5)

label_font_size = 12

for i, x in enumerate(np.linspace(0, 0.8, 5)):
    plt.axvline(x, color="gray", linestyle="--", lw=0.5, ymin=0.33)
    plt.annotate(f"{i}", xy=(x, -0.1), ha="center", fontsize=label_font_size)
plt.annotate("", xy=(0.0, 0.6), xytext=(0.2, 0.6), arrowprops=dict(arrowstyle="<->", shrinkA=0, shrinkB=0))
plt.annotate("Width of \n a sample", xy=(0.1, 0.7), ha="center", fontsize=label_font_size)
plt.annotate(
    "Signal is sampled at the\nbeginning of each interval", 
    xy=(0.0, -0.15), 
    xytext=(0.0, -0.4), 
    arrowprops=dict(arrowstyle="->", shrinkA=0, shrinkB=0), 
    fontsize=label_font_size
)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()


If we however combine multiple AGens with different sample rates, this can lead to inconsistencies.

In [ ]:
sig1 = Line(0, 1, 1, sr=10).gen_asig()
print(f"Last value of Line sampled with 10Hz: {sig1.sig[-1]}")
plt.show()
sig2 = Line(0, 1, 1, sr=1000).gen_asig()
print(f"Last value of Line sampled with 1000Hz: {sig2.sig[-1]}")

(Line(0, 0, 1, sr=1000) + Line(0, 1, 1, sr=10)).gen_asig().plot(label="10Hz upsampled to 1000Hz")
Line(0, 1, 1, sr=1000).gen_asig().plot(label="1000Hz")
plt.legend(prop={'size': 12});
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()

Here we see that the slope of the Line that was sampled with $1000\mathrm{Hz}$ is not the same as the slope of the Line sampled with $10\mathrm{Hz}$ and then upsampled to $1000\mathrm{Hz}$.
This happens because the last value of the Line sampled with $10\mathrm{Hz}$ is $0.9$ while the last value of the Line sampled with $1000\mathrm{Hz}$ is $0.999$.